# Sentiment Analysis & Returns Processing with CrewAI


## Overview

Jenny runs an online shoe shop — **Craved Rock Fitness Shoes** — and receives
dozens of customer comments every week. Reading through all of them manually
takes hours. She needs to know:

1. **How are customers feeling?** (positive, negative, neutral)
2. **What are they talking about?** (shipping, quality, sizing…)
3. **How should returns and refunds be handled automatically?**

In this notebook we build two CrewAI systems to solve all three problems.

### What you will learn

| Part | Topic | Key concepts |
|---|---|---|
| Part 1 | Sentiment analysis with custom tools | `Agent`, `Task`, `Crew`, `@tool` |
| Part 2 | CrewAI CLI project structure | `crew.py`, `agents.yaml`, `tasks.yaml` |
| Part 3 | Returns & refunds with CrewAI Flows | `Flow`, `@start`, `@router`, `@listen` |

> **Prerequisites:** Basic Python. No prior CrewAI experience needed.

---

**References**

- 📖 [CrewAI Documentation](https://docs.crewai.com)
- 📖 [CrewAI Agents](https://docs.crewai.com/concepts/agents)
- 📖 [CrewAI Tasks](https://docs.crewai.com/concepts/tasks)
- 📖 [CrewAI Crews](https://docs.crewai.com/concepts/crews)
- 📖 [CrewAI Tools](https://docs.crewai.com/concepts/tools)


---

## Getting Started with CrewAI

[CrewAI](https://docs.crewai.com) is an open-source Python framework for building
**multi-agent AI systems** — teams of AI agents that collaborate to complete tasks.

Think of it like a company:
- Each **Agent** is an employee with a job title, a goal, and a background
- Each **Task** is an assignment given to an agent
- A **Crew** is the team — it coordinates who does what and in what order

### Why use a framework like CrewAI?

You *could* build multi-agent systems by chaining LLM calls manually.
CrewAI handles the plumbing for you:
- Passing context from one agent to another
- Managing which tasks depend on which others
- Providing a clean, readable structure for your agent logic
- Scaling from a notebook experiment to a production CLI project


### CrewAI's Core Building Blocks

| Building Block | What it is | Defined by |
|---|---|---|
| **Agent** | An autonomous AI worker with a role, goal, and backstory | `Agent(role=..., goal=..., backstory=...)` |
| **Task** | A specific job for an agent, with a description and expected output | `Task(description=..., expected_output=..., agent=...)` |
| **Tool** | A Python function an agent can call to get real-world data | `@tool` decorator |
| **Crew** | The team — assembles agents and tasks, runs them in order | `Crew(agents=[...], tasks=[...])` |

The **backstory** is especially important — it gives the LLM context about
*who the agent is*, which shapes how it reasons and what tone it uses.
A "senior data analyst" agent will respond very differently from a
"friendly customer service rep" even with the same task.


---

## Sentiment Analysis with CrewAI Using Custom Tools

**Scenario:** Jenny has 12 customer comments. We build a CrewAI crew with
two agents that analyse the comments and produce a structured report.

We also use a **custom tool** — a Python function decorated with `@tool` —
to fetch the current date. This shows how agents can access real-time data.


In [1]:
# Upgrade pip to the latest version to avoid dependency resolution issues
%pip install -q --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Install CrewAI and python-dotenv (for loading the API key from .env)
%pip install -q crewai python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv

# load_dotenv() reads the .env file in the project root and sets environment variables
# This is how we securely pass the API key without hardcoding it in the notebook
load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not set — add it to your .env file"
print("✓ API key loaded successfully")


✓ API key loaded successfully


### Custom Tools — Giving Agents Real-World Access

An agent without tools can only use knowledge from its training data.
**Tools** let agents fetch live information — like today's date, stock prices,
or database records.

In CrewAI, a tool is just a Python function decorated with `@tool`.
The agent sees the function's **docstring** as the tool description —
that's how it decides when and how to use it.


In [4]:
from crewai.tools import tool
import datetime

@tool
def get_current_date() -> str:
    """Returns today's date and day of the week. Use this to add a timestamp to reports."""
    now = datetime.datetime.now()
    return f"{now.strftime('%A, %B %d, %Y')}"

# Test the tool
print(get_current_date.run(""))


TypeError: get_current_date() takes 0 positional arguments but 1 was given

### Sample Customer Feedback

These are Jenny's customer comments. Replace them with real feedback to analyze your own data.

In [ ]:
# Replace these with real customer comments to analyze your own data
CUSTOMER_FEEDBACK = """
1.  "The shoes arrived in 2 days — super fast! Great quality too."
2.  "Terrible experience. My order was wrong and support never responded."
3.  "Good shoes but the sizing runs small. Took two tries to get the right fit."
4.  "Love the styles! Will definitely order again."
5.  "Package arrived damaged. The shoes were fine but the box was crushed."
6.  "Best running shoes I've ever owned. Worth every penny."
7.  "Delivery took 3 weeks. Way too slow for the price I paid."
8.  "The customer service team was really helpful when I had a sizing question."
9.  "Shoes look exactly like the photos. Very happy with the purchase."
10. "Disappointed — the sole started peeling after just two weeks of use."
11. "Quick shipping, nice packaging, good product. Nothing to complain about."
12. "I asked for a refund 10 days ago and still haven't heard back."
"""

### Step 2 — Defining the Agents

We create two agents, each with a specific role:

| Agent | Role | Tool | What it produces |
|---|---|---|---|
| `sentiment_analyst` | Customer Sentiment Analyst | `get_current_date` | Sentiment % breakdown with representative quotes |
| `theme_analyst` | Feedback Theme Analyst | *(none)* | Top recurring themes with counts and quotes |

**Key agent fields:**
- **`role`** — the agent's job title; sets the tone and expertise of the LLM persona
- **`goal`** — what the agent is trying to achieve; guides its reasoning
- **`backstory`** — background story that shapes *how* the agent thinks and writes
- **`tools`** — optional list of functions the agent can call
- **`verbose=True`** — prints the agent's reasoning steps so you can follow along


In [ ]:
from crewai import Agent

sentiment_analyst = Agent(
    role="Customer Sentiment Analyst",
    goal=(
        "Analyze customer feedback comments and determine the overall sentiment "
        "distribution — what percentage of feedback is positive, negative, or neutral."
    ),
    backstory=(
        "You are an expert in natural language processing and consumer psychology. "
        "You have years of experience helping businesses understand how their customers "
        "feel about their products and services by reading between the lines of raw feedback."
    ),
    tools=[get_current_date],  # can timestamp the report
    verbose=True,
)

theme_analyst = Agent(
    role="Feedback Theme Analyst",
    goal=(
        "Identify the recurring topics and themes in customer feedback so the business "
        "owner knows exactly what areas customers are talking about most."
    ),
    backstory=(
        "You are a seasoned market researcher who specializes in thematic analysis. "
        "You have helped dozens of small businesses turn raw customer comments into "
        "clear, structured insights about what matters most to their customers."
    ),
    verbose=True,
)


### Step 3 — Defining the Tasks

A **Task** tells an agent exactly what to do and what a good result looks like.
Think of it as a detailed brief you'd give to a human employee.

Two fields are required:
- **`description`** — what to do (the full instructions)
- **`expected_output`** — what the result should look like (guides the LLM's formatting)

The `theme_task` has a **`context`** parameter pointing to `sentiment_task`.
This means the theme analyst sees the sentiment analyst's results before it runs —
it can build on that context rather than starting from scratch.

> **`context`** creates a dependency: `theme_task` waits for `sentiment_task` to finish first.


In [ ]:
from crewai import Task

sentiment_task = Task(
    description=(
        "Analyze the following customer feedback comments from Jenny's online shoe shop.\n"
        "For each comment, determine whether the sentiment is positive, negative, or neutral.\n"
        "Then calculate the overall sentiment breakdown as percentages.\n\n"
        f"{CUSTOMER_FEEDBACK}"
    ),
    expected_output=(
        "A clear sentiment breakdown report including:\n"
        "- Overall percentages: X% positive, Y% negative, Z% neutral\n"
        "- A brief explanation of what is driving each sentiment category\n"
        "- 2-3 representative quotes for each sentiment category"
    ),
    agent=sentiment_analyst,
)

theme_task = Task(
    description=(
        "Analyze the following customer feedback comments from Jenny's online shoe shop.\n"
        "Identify the main recurring themes and topics customers are discussing.\n"
        "Group similar mentions together and rank themes by how frequently they appear.\n\n"
        f"{CUSTOMER_FEEDBACK}"
    ),
    expected_output=(
        "A structured list of the top themes found in the feedback, each with:\n"
        "- Theme name (e.g. 'Shipping Speed', 'Product Quality', 'Customer Service')\n"
        "- How many comments mention it\n"
        "- A short description of what customers are saying about this theme\n"
        "- 1-2 representative quotes"
    ),
    agent=theme_analyst,
    context=[sentiment_task],  # theme analyst sees sentiment results first
)

### Step 4 — Assembling the Crew

The **Crew** is what ties everything together. It receives the list of agents
and tasks and manages the execution.

**Execution modes:**
- `Process.sequential` — tasks run one after the other in the order listed
- `Process.hierarchical` — a manager LLM decides which agent does what (more advanced)

We use `sequential` here: sentiment analysis runs first, then theme analysis
(which can use the sentiment results via `context`).


In [ ]:
from crewai import Crew, Process

crew = Crew(
    agents=[sentiment_analyst, theme_analyst],
    tasks=[sentiment_task, theme_task],
    process=Process.sequential,
    verbose=True,
)

### Step 5 — Running the Analysis

We call `await crew.kickoff_async()` instead of `crew.kickoff()` because
**Jupyter Notebook runs its own async event loop**. Calling the synchronous
version inside an existing loop raises a `RuntimeError`.

The `async` version avoids this completely — it's always safe to use in notebooks.

> **Expected output:** A detailed sentiment breakdown (% positive/negative/neutral
> with example quotes), followed by the top themes with descriptions and quotes.
> The agents' reasoning steps are printed above the final result because `verbose=True`.


In [ ]:
result = await crew.kickoff_async()
print(result)

---

## Part 2 — CrewAI CLI Project

The notebook approach is ideal for learning. For a **real application** you would
use the CrewAI CLI to scaffold a proper project — with configuration in YAML files,
a clean Python package structure, and a single command to run everything.

This also makes it easy to:
- **Change agent behaviour** without touching Python code (edit the YAML)
- **Test in isolation** — each agent/task is in its own file
- **Deploy** — the project is a standard Python package

The project lives at `feedback-analysis-crew/` in the repository root.

> **How to generate this structure yourself:**
> ```bash
> crewai create crew my-project-name
> ```


### Project Structure

A CrewAI CLI project generated with `crewai create crew feedback-analysis-crew` looks like this:

```
feedback-analysis-crew/
├── src/feedback_analysis_crew/
│   ├── crew.py               # Crew class — wires agents + tasks together
│   ├── main.py               # Entry point — called by `crewai run`
│   └── config/
│       ├── agents.yaml       # Agent definitions (role, goal, backstory)
│       └── tasks.yaml        # Task definitions (description, expected_output)
├── .env.example              # API key template
└── pyproject.toml            # Dependencies
```

The key difference from the notebook approach:
- Agents and tasks are defined in **YAML files**, not Python
- The `crew.py` class uses decorators (`@agent`, `@task`, `@crew`) to wire everything together
- You run the whole thing with a single terminal command: `crewai run`

### `agents.yaml` — Agent Definitions

This file defines your agents' roles, goals, and backstories — the same fields you set in Python, but in YAML for easier editing without touching code.

In [ ]:
# Read and display the agents.yaml file from the CLI project
agents_yaml_path = "../../feedback-analysis-crew/src/feedback_analysis_crew/config/agents.yaml"

with open(agents_yaml_path) as f:
    print(f.read())

### `tasks.yaml` — Task Definitions

This file defines what each agent must do. Notice the `{feedback}` placeholder — at runtime it gets replaced with the actual customer comments passed via `crew.kickoff(inputs={'feedback': ...})`.

In [ ]:
tasks_yaml_path = "../../feedback-analysis-crew/src/feedback_analysis_crew/config/tasks.yaml"

with open(tasks_yaml_path) as f:
    print(f.read())

### `crew.py` — The Crew Class

The `@CrewBase` decorator and `@agent` / `@task` / `@crew` decorators wire the YAML config to Python objects automatically. This is the glue between your config files and the CrewAI runtime.

In [ ]:
crew_py_path = "../../feedback-analysis-crew/src/feedback_analysis_crew/crew.py"

with open(crew_py_path) as f:
    print(f.read())

### `main.py` — Entry Point

This is what runs when you type `crewai run` in the terminal. It passes the feedback as an input — exactly like calling `crew.kickoff(inputs={...})` in the notebook.

In [ ]:
main_py_path = "../../feedback-analysis-crew/src/feedback_analysis_crew/main.py"

with open(main_py_path) as f:
    print(f.read())

### Running the CLI Project from Terminal

```bash
# Navigate to the project
cd feedback-analysis-crew

# Copy the env template and add your API key
cp .env.example .env
# Edit .env and set OPENAI_API_KEY=your-key

# Install dependencies
pip install crewai

# Run the crew
python src/feedback_analysis_crew/main.py
```

Or if you have the full CrewAI toolchain installed:

```bash
crewai run
```

---

## Notebook vs CLI Project

| | Notebook (Part 1) | CLI Project (Part 2) |
|---|---|---|
| **Best for** | Learning, experimenting, demos | Production apps, real projects |
| **Agents & Tasks** | Defined inline in Python | Defined in YAML config files |
| **Run with** | `await crew.kickoff_async()` | `crewai run` or `python main.py` |
| **Config changes** | Edit the notebook cell | Edit `agents.yaml` / `tasks.yaml` — no Python changes |
| **Structure** | Single file | Proper package with `src/`, `config/` |

**The logic is identical** — the only difference is where you put the config.

## Working with CrewAI Flows

**CrewAI Flows** are the next level after Crews. While a Crew has agents collaborate
on a single goal, a Flow orchestrates multiple Crews into a structured pipeline —
with sequential steps, conditional branching, and shared state.

### Key Decorators

| Decorator | Purpose |
|---|---|
| `@start()` | Marks the entry point — first method to run |
| `@listen(method)` | Runs automatically after the specified method completes |
| `@router(method)` | Branches to different `@listen` paths based on a return value |

### Shared State

Flows carry a **state object** (a Pydantic model) that every step can read and write.
This is how data flows from one Crew to the next without manual passing.

```python
from pydantic import BaseModel
from crewai.flow.flow import Flow, listen, router, start

class MyState(BaseModel):
    input: str = ""
    result: str = ""

class MyFlow(Flow[MyState]):
    @start()
    def step_one(self):
        self.state.result = SomeCrew().crew().kickoff(
            inputs={"input": self.state.input}
        ).raw

    @listen(step_one)
    def step_two(self):
        # runs automatically after step_one
        ...
```

**References**

- 📖 [CrewAI Flows Documentation](https://docs.crewai.com/concepts/flows)


## Developing an Agentic System with CrewAI Flows for Returns and Refunds

In the previous section we learned how Flows work. Now we build a complete, real-world
Flow for handling customer return requests. The full project lives at
`returns-refunds-flow/` in the repository root.


### Project Structure

```
returns-refunds-flow/
├── src/returns_refunds_flow/
│   ├── flow.py               # The Flow class — @start, @router, @listen steps
│   ├── crews.py              # Three Crew classes: Validation, Refund, Denial
│   ├── main.py               # Entry point with sample return requests
│   └── config/
│       ├── agents.yaml       # return_validator, refund_processor, denial_agent
│       └── tasks.yaml        # validate, process_refund, explain_denial tasks
├── .env.example
└── pyproject.toml
```

The key new file is **`flow.py`** — it replaces `crew.py` as the top-level orchestrator.
The Crews become building blocks that the Flow calls in sequence.

### How the Flow Works

The `ReturnsRefundsFlow` processes a customer return request in three steps:

```
         [START]
            │
       validate()
    ValidationCrew decides
    APPROVED or REJECTED
            │
      route_decision()
       @router branches
       ┌──────┴──────┐
  "approved"      "rejected"
       │                │
 process_approved()  process_rejected()
   RefundCrew         DenialCrew
  gives customer    gives empathetic
  return steps      denial + alternatives
```

The **shared state** (`ReturnState`) carries data between steps:
- `return_request` — the customer's original message (set before kickoff)
- `validation_result` — the validator's full analysis
- `decision` — `"APPROVED"` or `"REJECTED"`
- `final_response` — the message sent back to the customer

### `flow.py` — The Flow Class

This is the heart of the project. Notice:
- `ReturnState` is a **Pydantic model** — typed, validated, shared across all steps
- `@start()` kicks off the validation crew
- `@router` reads the decision and returns a string (`"approved"` or `"rejected"`)
- `@listen("approved")` and `@listen("rejected")` each trigger the right crew

In [ ]:
flow_py_path = "../../returns-refunds-flow/src/returns_refunds_flow/flow.py"

with open(flow_py_path) as f:
    print(f.read())

### `crews.py` — Three Crews as Building Blocks

Instead of one large Crew, we split responsibilities across three small Crews.
Each Crew has a single agent and a single task — focused, testable, reusable.

In [ ]:
crews_py_path = "../../returns-refunds-flow/src/returns_refunds_flow/crews.py"

with open(crews_py_path) as f:
    print(f.read())

### `agents.yaml` — Three Specialized Agents

Each agent in the flow has a single, focused responsibility:

| Agent key | Role | Used in |
|---|---|---|
| `return_validator` | Decides if the return is eligible | `ValidationCrew` |
| `refund_processor` | Generates return instructions | `RefundCrew` (approved path) |
| `denial_agent` | Writes an empathetic denial | `DenialCrew` (rejected path) |


In [ ]:
agents_yaml_path = "../../returns-refunds-flow/src/returns_refunds_flow/config/agents.yaml"

with open(agents_yaml_path) as f:
    print(f.read())

### `tasks.yaml` — One Task per Crew

Notice that `process_refund_task` and `explain_denial_task` both accept
`{validation_result}` as an input — the Flow passes the validator's output
to whichever downstream crew runs.

In [ ]:
tasks_yaml_path = "../../returns-refunds-flow/src/returns_refunds_flow/config/tasks.yaml"

with open(tasks_yaml_path) as f:
    print(f.read())

### `main.py` — Entry Point with Three Test Scenarios

Three scenarios are included so you can test all paths:
- `eligible` — a valid return within 30 days, unworn
- `ineligible` — a final-sale item returned after 45 days
- `defective` — a defective item (always approved regardless of condition)

In [ ]:
main_py_path = "../../returns-refunds-flow/src/returns_refunds_flow/main.py"

with open(main_py_path) as f:
    print(f.read())

### Running the Flow from Terminal

```bash
cd returns-refunds-flow

# Set up your API key
cp .env.example .env
# Edit .env: OPENAI_API_KEY=your-key

# Run with the default scenario (eligible)
python src/returns_refunds_flow/main.py

# Run a specific scenario
python src/returns_refunds_flow/main.py ineligible
python src/returns_refunds_flow/main.py defective
```

---

## Crew vs Flow — When to Use Which

| | Crew | Flow |
|---|---|---|
| **Best for** | Agents collaborating on one goal | Multi-step pipelines with branching |
| **Structure** | Agents + Tasks | Steps connected with `@start` / `@listen` / `@router` |
| **Branching** | No | Yes — `@router` routes to different `@listen` methods |
| **State** | Not built-in | Typed Pydantic model shared across all steps |
| **Complexity** | Lower | Higher — use when workflow logic is needed |
| **Example** | Analyze 12 comments in parallel | Validate → route → approve or deny |